Table MASContact_Contacts with all contact information associated with an MAS contract

In [0]:
from pyspark.sql import functions as F

#fields of interest: "an_emails" Authorized negotiator emails as a concatanated list; "vendor_email" Vendor email; "e_mail_adrs"- Buyer/CS email; "e_mail_adrs2"- Contractor/vendor email address (how does this differ from vendor_email?); "formal_cont"- short contract number, "phone_no"- vendor phone number; "duna_poc" - Administrative Point of Contact (POC); 

fss19_contBase = "/Volumes/fas_eda_fss19_cmf_prd/gold/fss19_contract_master"

folders = dbutils.fs.ls(fss19_contBase)
latest_folder = sorted([f.path for f in folders], reverse=True)[0]
print(f"FSS19 Contract Latest folder: {latest_folder}")

# load only the latest folder, apply CMF business rules
fss19_contractContacts = (
    spark.read.parquet(latest_folder)
    .withColumn("cont_end_dt", F.when(F.col("cont_end_dt") != "", F.col("cont_end_dt").cast("string")).otherwise(None))
    .withColumn("cont_beg_dt", F.when(F.col("cont_beg_dt") != "", F.col("cont_beg_dt").cast("string")).otherwise(None))
    .withColumn("dt_term", F.when(F.col("dt_term") != "", F.col("dt_term").cast("string")).otherwise(None))
    .filter(F.col("cont_ind") == "F")
    .select(
        F.col("gsam_cont_no"),
        F.col("f_cont_no"),
        F.col("uei"),
        F.col("legal_bus_name"),
        F.col("koca"),
        F.col("cont_beg_dt"), 
        F.col("cont_end_dt"),
        F.col("dt_term"), 
        F.col("phone_no"), 
        F.col("duna_poc"),
        F.col("an_emails"),
        F.col("vendor_email"),
        F.col("e_mail_adrs2")
    )        
    )

#add new column "is_current" 1= yes 0=no
#.filter(~F.col("koca").isin(["8", "9"])),  .filter(F.col("cont_end_dt") > F.current_date()), dt_term is not null or ""
fss19_contractContacts =fss19_contractContacts.withColumn("is_current", F.when(~F.col("koca").isin(["8", "9"]) | (F.col("cont_end_dt") > F.current_date()) | (F.col("dt_term").isNotNull()) | (F.col("dt_term") != ""), 1).otherwise(0))

display(fss19_contractContacts.limit(10))
print(f"count of fss19_contractContacts: {fss19_contractContacts.count()}")


# save to csv /Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/FSSContracts/
fss19_contractContacts.write.mode("overwrite").option("header", "true").csv("/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/FSSContracts/fss19_contractContacts.csv")

In [0]:
%skip
#cannot use this table for pocs because it contains the complete history of pocs, and not just the current ones.

from pyspark.sql import functions as F
#fields of interest: "aco_email" ACO email; "aco_name" ACO name; "aut_ng_name" Authorized negotiator name; "aut_ng_email" Authorized negotiator email;

from pyspark.sql import functions as F

fss19_modBase = "/Volumes/fas_eda_fss19_contract_modification_prd/gold/fss19_contract_modification"

folders = dbutils.fs.ls(fss19_modBase)
import re

date_pattern = re.compile(r"\d{4}-\d{2}-\d{2}$")
folders_date = [f.path for f in folders if date_pattern.search(f.path.rstrip("/").split("/")[-1])]
latest_folder = sorted(folders_date, reverse=True)[0]
# latest_folder = sorted([f.path for f in folders], reverse=True)[0]
print(f"FSS19 Modification Latest folder: {latest_folder}")

# load only the latest folder, apply CMF business rules
fss19_modContacts = (
    spark.read.parquet(latest_folder)
    .select(
        F.col("f_cont_no"),
        F.col("aco_name"),
        F.col("aco_email"),
        F.col("aut_ng_name"),
        F.col("aut_ng_email")
    ) 
    .orderBy(F.col("f_cont_no").asc())       
    )
print(f"fss19_modContacts count: {fss19_modContacts.count()}")
display(fss19_modContacts.limit(2))

#most amount of auth_ng_email per contract
auth_ng_name_count_df = (
    fss19_modContacts.groupBy("f_cont_no")
    .agg(F.count("aut_ng_name").alias("auth_ng_name_count"))
)

display(auth_ng_name_count_df.orderBy(F.col("auth_ng_name_count").desc()))

highest_auth_ng_name_count = auth_ng_name_count_df.agg(F.max("auth_ng_name_count").alias("max_auth_ng_name_count"))
display(highest_auth_ng_name_count)

#median auth_ng_name_count
median_auth_ng_name_count = auth_ng_name_count_df.agg(F.percentile_approx("auth_ng_name_count", [0.5]).alias("median_auth_ng_name_count"))
display(median_auth_ng_name_count)

#save to csv 
fss19_modContacts.write.mode("overwrite").option("header", "true").csv("/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/FSSContracts/fss19_modContacts.csv")

Use the poc_type to determine which fields the name and email address go under. 

"poc_type"
- va_contract_specialist -VA contract CS
- va_sales - Va contract maybe contractor sales person
- iff - MAS contractor maybe IFF POC, do they get emails?
- adminrep -MAS contractor maybe POC, sames as dunsPOC?
- ioa -IOA 
- aco- ACO
- va_primary -VA contractor person
- va_alternate - VA contractor person
- authorizednegotiator - MAS contractor authorized negotiator
- pcs -MAS CS
- pco -MAS CO

In [0]:
%skip
from pyspark.sql import functions as F
#srp_clean_poc_all_contracts

srp_cleanBase = "/Volumes/fas_eda_srp_contract_data_prd/gold/srp_clean_poc_all_contracts"

folders = dbutils.fs.ls(srp_cleanBase)
latest_folder = sorted([f.path for f in folders], reverse=True)[0]
print(f"FSS19 Contract Latest folder: {latest_folder}")

# load only the latest folder, apply CMF business rules
srp_cleanContacts = (
    spark.read.parquet(latest_folder)
)
display(srp_cleanContacts.limit(10))
print(f"count of srp_cleanContacts: {srp_cleanContacts.count()}")

srp_cleanContacts_pocType_df = srp_cleanContacts.select(F.col("poc_type")).distinct()
display(srp_cleanContacts_pocType_df)
print(f"count of srp_cleanContacts_pocType_df: {srp_cleanContacts_pocType_df.count()}")

#what are the "poc_type" va_contract_specialist, va_sales, iff, adminrep, va_primary, va_alternate
print("va_contract_specialist")
srp_cleanContacts_va_cs_df =srp_cleanContacts.filter(F.col("poc_type") == "va_contract_specialist")
display(srp_cleanContacts_va_cs_df.limit(5))
print(f"count of srp_cleanContacts_va_cs_df: {srp_cleanContacts_va_cs_df.count()}")

print("va_sales")
srp_cleanContacts_va_sales_df =srp_cleanContacts.filter(F.col("poc_type") == "va_sales")
display(srp_cleanContacts_va_sales_df.limit(5))
print(f"count of srp_cleanContacts_va_sales_df: {srp_cleanContacts_va_sales_df.count()}")

print("iff")
srp_cleanContacts_iff_df =srp_cleanContacts.filter(F.col("poc_type") == "iff")
display(srp_cleanContacts_iff_df.limit(5))
print(f"count of srp_cleanContacts_iff_df: {srp_cleanContacts_iff_df.count()}")

print("adminrep")
srp_cleanContacts_adminrep_df =srp_cleanContacts.filter(F.col("poc_type") == "adminrep")
display(srp_cleanContacts_adminrep_df.limit(5))
print(f"count of srp_cleanContacts_adminrep_df: {srp_cleanContacts_adminrep_df.count()}")

print("va_primary")
srp_cleanContacts_va_primary_df =srp_cleanContacts.filter(F.col("poc_type") == "va_primary")
display(srp_cleanContacts_va_primary_df.limit(5))
print(f"count of srp_cleanContacts_va_primary_df: {srp_cleanContacts_va_primary_df.count()}")

print("va_alternate")
srp_cleanContacts_va_alternate_df =srp_cleanContacts.filter(F.col("poc_type") == "va_alternate")
display(srp_cleanContacts_va_alternate_df.limit(5))
print(f"count of srp_cleanContacts_va_alternate_df: {srp_cleanContacts_va_alternate_df.count()}")



In [0]:
from pyspark.sql import functions as F
#srp_clean_poc_all_contracts

srp_cleanBase = "/Volumes/fas_eda_srp_contract_data_prd/gold/srp_clean_poc_all_contracts"

folders = dbutils.fs.ls(srp_cleanBase)
latest_folder = sorted([f.path for f in folders], reverse=True)[0]
print(f"FSS19 Contract Latest folder: {latest_folder}")

# load only the latest folder, apply CMF business rules
srp_cleanContacts = (
    spark.read.parquet(latest_folder)
    .filter(~F.col("poc_type").isin("va_contract_specialist", "va_sales", "va_primary","va_alternate"))
)
display(srp_cleanContacts.limit(10))
print(f"count of srp_cleanContacts: {srp_cleanContacts.count()}")

# save to csv /Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/FSSContracts/
srp_cleanContacts.write.mode("overwrite").option("header", "true").csv("/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/FSSContracts/srp_cleanContacts.csv")

In [0]:
%skip
from pyspark.sql import functions as F
#Does the srp_cleanContacts table only contain current values? 

# Bring in srp_cleanContacts
srp_cleanContacts_path = "/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/FSSContracts/srp_cleanContacts.csv"
srp_cleanContacts_df = spark.read.csv(srp_cleanContacts_path, header=True)

#how many IOAs are there per contract? "ioa"
srp_cleanContracts_IOA_df = srp_cleanContacts_df.filter(F.col("poc_type") == "ioa")
print(f"count of srp_cleanContacts_va_cs_df: {srp_cleanContracts_IOA_df .count()}")

#distinct contract_id
srp_cleanContracts_distinct_df = srp_cleanContracts_IOA_df.select(F.col("contract_id")).distinct()
print(f"count of srp_cleanContracts_distinct_df: {srp_cleanContracts_distinct_df.count()}")

#both are 51264, so only 1 IOA per contract

#How many Authorized Negotiators are there per contract? "authorizednegotiator"
srp_cleanContracts_auth_neg_df = srp_cleanContacts_df.filter(F.col("poc_type") == "authorizednegotiator")
print(f"count of srp_cleanContracts_auth_neg_df: {srp_cleanContracts_auth_neg_df.count()}")
#rename column "poc_type" to "authorizednegotiator"
srp_cleanContracts_auth_neg_df = srp_cleanContracts_auth_neg_df.withColumnRenamed("poc_type", "authorizednegotiator")


auth_ng_count_df = (
    srp_cleanContracts_auth_neg_df.groupBy("contract_id")
    .agg(F.count("authorizednegotiator").alias("auth_ng_count_df"))
)

display(auth_ng_count_df.orderBy(F.col("auth_ng_count_df").desc()).limit(10))

highest_auth_ng_count_df = auth_ng_count_df.agg(F.max("auth_ng_count_df").alias("max_auth_ng_count_df"))
display(highest_auth_ng_count_df)

#median auth_ng_count_df
median_auth_ng_count_df = auth_ng_count_df.agg(F.percentile_approx("auth_ng_count_df", [0.5]).alias("median_auth_ng_count_df"))
display(median_auth_ng_count_df)


#Maximum # of authorized negotiators per contract is 19, median is 2



Join Fss19_contract Master File with srp_cleanContacts, by adding a column for each poc type and their email to fss19_contractContacts, and then going row by row of srp_cleanContacts and matching on contract then filling appropriate column. 

Since there are multiple Authorized negotiators


In [0]:
%skip
from pyspark.sql import functions as F

#redone in next node
#bring in srp_cleanContacts
srp_cleanContacts_path = "/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/FSSContracts/srp_cleanContacts.csv"
srp_cleanContacts_df = spark.read.csv(srp_cleanContacts_path, header=True)

#Construct a table with columns for each of the poc_types and emails for that poc type
#poc_types
# iff
# adminrep
# ioa
# aco
# authorizednegotiator
# pcs
# pco

#create distinct contract_id table
poc_cleanContracts_df = srp_cleanContacts_df.select(F.col("contract_id")).distinct()
poc_cleanContracts_df = poc_cleanContracts_df.withColumnRenamed("contract_id","f_cont_no")

#add columns for each poc_type and email, concatenate authorized negotiator values
poc_cleanContracts_df = poc_cleanContracts_df.withColumns({"iff_name": F.lit(None), "iff_email": F.lit(None), "adminrep_name": F.lit(None), "adminrep_email": F.lit(None), "ioa_name": F.lit(None), "ioa_email": F.lit(None), "aco_name": F.lit(None), "aco_email": F.lit(None), "authorizednegotiator_name": F.lit(None), "authorizednegotiator_email": F.lit(None), "pcs_name": F.lit(None), "pcs_email": F.lit(None), "pco_name": F.lit(None), "pco_email": F.lit(None)})

#First fill in authorized negotiator information and concatenate
#when srp_cleanContacts_df "contract_id" == poc_cleanContacts_df "F_cont_no", & srp_cleanContacts_df "poc_type" == "authorizednegotiator", then concatenate and fill in srp_cleanContacts_df "email" as poc_cleanContacts_df "authorizednegotiator_email" & concatenate and fill srp_cleanContacts_df "name" as poc_cleanContacts_df "authorizednegotiator_name"
# poc_cleanContracts_df = srp_cleanContacts_df.join(poc_cleanContracts_df, F.when(srp_cleanContacts_df["contract_id"] == poc_cleanContracts_df["f_cont_no"], F.concat_ws(",", F.collect_list(F.when(srp_cleanContacts_df("poc_type") == "authorizednegotiator", srp_cleanContacts_df("email") ==  poc_cleanContracts_df("authorizednegotiator_email"))))).over(Window.partitionBy("f_cont_no")) & F.when(srp_cleanContacts_df["contract_id"] == poc_cleanContracts_df["f_cont_no"], F.concat_ws(",", F.collect_list(F.when(srp_cleanContacts_df("poc_type") == "authorizednegotiator", srp_cleanContacts_df("name") ==    poc_cleanContracts_df("authorizednegotiator_name"))))).over(Window.partitionBy("f_cont_no")),how="inner")

from pyspark.sql.window import Window

# Window for partitioning by contract
contract_window = Window.partitionBy("contract_id")

# Aggregate authorized negotiator emails and names
auth_neg_email_df = srp_cleanContacts_df.filter(F.col("poc_type") == "authorizednegotiator") \
    .groupBy("contract_id") \
    .agg(F.concat_ws(",", F.collect_list("email")).alias("authorizednegotiator_email"),
         F.concat_ws(",", F.collect_list("name")).alias("authorizednegotiator_name"))

# Aggregate iff emails and names (only one per contract, so take first)
iff_df = srp_cleanContacts_df.filter(F.col("poc_type") == "iff") \
    .groupBy("contract_id") \
    .agg(F.first("email").alias("iff_email"),
         F.first("name").alias("iff_name"))

#Aggregate adminrep emails and names
adminrep_df = srp_cleanContacts_df.filter(F.col("poc_type") == "adminrep") \
    .groupBy("contract_id") \
    .agg(F.first("email").alias("adminrep_email"),
         F.first("name").alias("adminrep_name"))

#Aggregate ioa emails and names
ioa_df = srp_cleanContacts_df.filter(F.col("poc_type") == "ioa") \
    .groupBy("contract_id") \
    .agg(F.first("email").alias("ioa_email"),
         F.first("name").alias("ioa_name"))
    
#Aggregate aco emails and names
aco_df = srp_cleanContacts_df.filter(F.col("poc_type") == "aco") \
    .groupBy("contract_id") \
    .agg(F.first("email").alias("aco_email"),
         F.first("name").alias("aco_name"))
#Aggregate pcs emails and names
pcs_df = srp_cleanContacts_df.filter(F.col("poc_type") == "pcs") \
    .groupBy("contract_id") \
    .agg(F.first("email").alias("pcs_email"),
         F.first("name").alias("pcs_name"))

#Aggregate pco emails and names
pco_df = srp_cleanContacts_df.filter(F.col("poc_type") == "pco") \
    .groupBy("contract_id") \
    .agg(F.first("email").alias("pco_email"),
         F.first("name").alias("pco_name"))
    
    
# Join aggregated columns to poc_cleanContracts_df
# poc_cleanContracts_df = poc_cleanContracts_df \
#     .join(auth_neg_email_df, poc_cleanContracts_df["f_cont_no"] == auth_neg_email_df["contract_id"], "left") \
#     .join(iff_df, poc_cleanContracts_df["f_cont_no"] == iff_df["contract_id"], "left") \
#     .drop(auth_neg_email_df["contract_id"]) \
#     .drop(iff_df["contract_id"])

#Second fill in iff information and concatenate






#     poc_cleanContacts_df = poc_cleanContacts_df.withColumn("iff_email", F.when(F.col("poc_type") == "iff", F.col("email"))).over(Window.partitionBy("f_cont_no"))\npoc_cleanContacts_df = poc_cleanContacts_df.withColumn("iff_name", F.when(F.col("poc_type") == "iff", F.col("name")))\n    .over(Window.partitionBy("f_cont_no"))
#      poc_cleanContacts_df = poc_cleanContacts_df.withColumn("authorizednegotiator", F.concat_ws(",", F.collect_list(F.when(F.col("poc_type") == "authorizednegotiator", F.col("email"))))).over(Window.partitionBy("f_cont_no"))
    
# })

display(poc_cleanContracts_df.limit(10))
  # "iff": F.when(F.col("poc_type") == "iff", F.col("email").alias("iff_email")).over(Window.partitionBy("f_cont_no")) && F.when(F.col("poc_type") == "iff", F.col("name").alias("iff_name")).over(Window.partitionBy("f_cont_no")),
# "adminrep": F.concat_ws(",", F.collect_list(F.when(F.col("poc_type") == "adminrep", F.col("email")).otherwise(F.lit(""))).over(Window.partitionBy("f_cont_no"))),
#     "ioa": F.concat_ws(",", F.collect_list(F.when(F.col("poc_type") == "ioa", F.col("email")).otherwise(F.lit(""))).over(Window.partitionBy("f_cont_no"))),
#     "aco": F.concat_ws(",", F.collect_list(F.when(F.col("poc_type") == "aco", F.col("email")).otherwise(F.lit(""))).over(Window.partitionBy("f_cont_no"))),
   
#     "pcs": F.concat_ws(",", F.collect_list(F.when(F.col("poc_type") == "pcs", F.col("email")).otherwise(F.lit(""))).over(WindowBy.partitionBy("f_cont_no"))),
#     "pco": F.concat_ws(",", F.collect_list(F.when(F.col("poc_type") == "pco", F.col("email")).otherwise(F.lit(""))).over(Window.partitionBy("f_cont_no")))



In [0]:
from pyspark.sql import functions as F

#bring in srp_cleanContacts
srp_cleanContacts_path = "/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/FSSContracts/srp_cleanContacts.csv"
srp_cleanContacts_df = spark.read.csv(srp_cleanContacts_path, header=True)

#Construct a table with columns for each of the poc_types and emails for that poc type
#poc_types
# iff
# adminrep
# ioa
# aco
# authorizednegotiator
# pcs
# pco

from pyspark.sql.window import Window

# Window for partitioning by contract
contract_window = Window.partitionBy("contract_id")

# Aggregate authorized negotiator emails and names, because there are multiple per contract. Max per contract is 19
auth_neg_email_df = srp_cleanContacts_df.filter(F.col("poc_type") == "authorizednegotiator") \
    .groupBy("contract_id") \
    .agg(F.concat_ws(",", F.collect_list("email")).alias("authorizednegotiator_email"),
         F.concat_ws(",", F.collect_list("name")).alias("authorizednegotiator_name"))

# Aggregate iff emails and names (only one per contract, so take first)
iff_df = srp_cleanContacts_df.filter(F.col("poc_type") == "iff") \
    .groupBy("contract_id") \
    .agg(F.first("email").alias("iff_email"),
         F.first("name").alias("iff_name"))

#Aggregate adminrep emails and names
adminrep_df = srp_cleanContacts_df.filter(F.col("poc_type") == "adminrep") \
    .groupBy("contract_id") \
    .agg(F.first("email").alias("adminrep_email"),
         F.first("name").alias("adminrep_name"))

#Aggregate ioa emails and names
ioa_df = srp_cleanContacts_df.filter(F.col("poc_type") == "ioa") \
    .groupBy("contract_id") \
    .agg(F.first("email").alias("ioa_email"),
         F.first("name").alias("ioa_name"))
    
#Aggregate aco emails and names
aco_df = srp_cleanContacts_df.filter(F.col("poc_type") == "aco") \
    .groupBy("contract_id") \
    .agg(F.first("email").alias("aco_email"),
         F.first("name").alias("aco_name"))
#Aggregate pcs emails and names
pcs_df = srp_cleanContacts_df.filter(F.col("poc_type") == "pcs") \
    .groupBy("contract_id") \
    .agg(F.first("email").alias("pcs_email"),
         F.first("name").alias("pcs_name"))

#Aggregate pco emails and names
pco_df = srp_cleanContacts_df.filter(F.col("poc_type") == "pco") \
    .groupBy("contract_id") \
    .agg(F.first("email").alias("pco_email"),
         F.first("name").alias("pco_name"))
    

# Join aggregated columns and save as poc_cleanContracts_df
from functools import reduce

# Store all dataframes in a list
dfs = [auth_neg_email_df, iff_df, adminrep_df, ioa_df, aco_df, pcs_df, pco_df]

# Iteratively join all dataframes on the "id" column
poc_cleanContracts_df = reduce(lambda left, right: left.join(right, on="contract_id", how="inner"), dfs)

display(poc_cleanContracts_df.limit(10))
print(f"There are {poc_cleanContracts_df.count()} rows in poc_cleanContracts_df")


# save to csv /Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/FSSContracts/
poc_cleanContracts_df.write.mode("overwrite").option("header", "true").csv("/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/FSSContracts/poc_cleanContracts_df.csv")

In [0]:
#Complete new dataFrame by joining poc_cleanContracts_df and fss19_contractContacts

#Bring in dataframes
poc_cleanContracts_path = "/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/FSSContracts/poc_cleanContracts_df.csv"
poc_cleanContracts_df = spark.read.csv(poc_cleanContracts_path, header=True)

#rename contract_id
poc_cleanContracts_df = poc_cleanContracts_df.withColumnRenamed("contract_id", "gsam_cont_no")

fss19_contractContacts_path = "/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/FSSContracts/fss19_contractContacts.csv"
fss19_contractContacts_df = spark.read.csv(fss19_contractContacts_path, header=True)

#join the two dataframes with a left join so only contracts from fss19_contractContacts_df are kept
MASContract_Contacts = fss19_contractContacts_df.join(poc_cleanContracts_df, on="gsam_cont_no", how="left")

display(MASContract_Contacts.limit(10))
print(f"There are {MASContract_Contacts.count()} rows in MASContract_Contacts")


#save to csv /Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/FSSContracts/
MASContract_Contacts.write.mode("overwrite").option("header", "true").csv("/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/FSSContracts/MASContract_Contacts.csv")

#Also save to shared Volume: /Volumes/fas_eda_analytics_prd/default/create_public
MASContract_Contacts.write.mode("overwrite").option("header", "true").csv("/Volumes/fas_eda_analytics_prd/default/create_public/MASContract_Contacts.csv")